In [4]:
import pandas as pd
import numpy as np
import os
import random

# ================= 1. 配置区 =================
RAW_KG_PATH = "kg.csv"
OUTPUT_PATH = "primekg_nacc_base.csv"

BLACKLIST_NODES = [
    # --- 解剖黑名单 ---
    "multi-cellular organism", "small intestine", "testis", "fallopian tube", "prostate gland", 
    "spleen", "intestine", "esophagus", "stomach", "colon",
    "adrenal gland", "saliva-secreting gland", "myometrium", "female gonad", 
    "urinary bladder", "endometrium", "lymph node", 
    "dorsolateral prefrontal cortex", "prefrontal cortex", "frontal cortex", "cerebellar cortex",
    "heart left ventricle", "heart", "cardiac muscle",
    "thyroid gland", "adult mammalian kidney", "cortex of kidney", 
    "pancreas", "blood", "muscle of leg",
    
    # --- [新增] 细胞组分与宽泛概念 ---
    "cytoplasm", "cytosol", "nucleus", "membrane", "plasma membrane", 
    "extracellular region", "extracellular space", "mitochondrion",
    "protein binding", "molecular_function", "biological_process", "cellular_component",
    "cell surface", "intracellular", "nucleoplasm"
]

# 【关键词】(保持精简版)
KEYWORDS = [
    # AD
    "Alzheimer", "Dementia", "Memory", "Cognitive", "Amyloid", "Tau", "Neurofibrillary",
    # 核心代谢 (Hypertension, Hypercholesterolemia)
    "Hypertension", "Blood Pressure",
    "Cholesterol", "Lipid", "Lipoprotein", "Hyperlipidemia",
    # 精神 (Depression)
    "Depression", "Depressive", "Mood",
    # 其他 (Thyroid, Diabetes, Anxiety)
    "Thyroid", "Diabetes", "Insulin", "Glucose", "Anxiety"
]

# 【PPI 限制参数】这是瘦身的关键！
MAX_PPI_DEGREE = 5  # 每个基因最多保留 5 条 PPI 边

def main():
    print(f"🔹 1. 加载 PrimeKG...")
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    except:
        print("❌ 找不到文件")
        return

    # ================= 步骤 A: 基础清洗 =================
    print(f"🔹 2. 执行清洗 (含新增的细胞组分黑名单)...")
    valid_types = ['disease', 'gene/protein', 'effect/phenotype', 'pathway', 'biological_process', 'anatomy']
    
    mask = (
        (df['x_type'].isin(valid_types)) & 
        (df['y_type'].isin(valid_types)) &
        (~df['relation'].isin(['indication', 'contraindication', 'drug_drug', 'off-label use'])) &
        (~df['x_name'].isin(BLACKLIST_NODES)) & 
        (~df['y_name'].isin(BLACKLIST_NODES))
    )
    df_clean = df[mask].copy()
    print(f"   🧹 基础清洗后: {len(df_clean)}")

    # ================= 步骤 B: 关键词筛选 (核心知识) =================
    print(f"🔹 3. 筛选 NACC 核心特征...")
    pattern = '|'.join(KEYWORDS)
    mask_kw = (df_clean['x_name'].astype(str).str.contains(pattern, case=False, na=False)) | \
              (df_clean['y_name'].astype(str).str.contains(pattern, case=False, na=False))
    
    core_df = df_clean[mask_kw].copy()
    print(f"   ✅ 核心知识 (AD/代谢/精神): {len(core_df)} 条")
    
    # ================= 步骤 C: PPI 智能瘦身 (关键步骤) =================
    print(f"\n🔹 4. 补充基因互作 (限制每个基因最多 {MAX_PPI_DEGREE} 条边)...")
    
    # 1. 拿到核心知识里的所有基因
    genes = set(core_df[core_df['y_type'] == 'gene/protein']['y_name']) | \
            set(core_df[core_df['x_type'] == 'gene/protein']['x_name'])
    
    # 2. 提取子图 PPI
    mask_ppi = (
        (df_clean['x_name'].isin(genes)) &
        (df_clean['y_name'].isin(genes)) &
        (df_clean['relation'] == 'protein_protein')
    )
    ppi_full = df_clean[mask_ppi].copy()
    print(f"   🕸  原始全量 PPI: {len(ppi_full)} 条 (太多了，准备剪枝)")
    
    # 3. 执行度数限制剪枝
    # 逻辑：打乱顺序 -> 按基因分组 -> 每个基因只取前 K 个
    # 这样能保证覆盖所有基因，但削减稠密连接
    ppi_full = ppi_full.sample(frac=1, random_state=42) # 打乱
    
    # 简单的贪心保留法：
    # 我们遍历每一条边，记录每个节点已保留的度数。如果两端都满了，就丢弃。
    keep_indices = []
    node_degrees = {g: 0 for g in genes}
    
    # 优先保留连接“AD核心基因”的边 (可选优化，这里先做通用的)
    # 我们可以把含 "Alzheimer" 相关的基因设为 VIP，允许更高度数，但为了简单，先统一限制
    
    for idx, row in ppi_full.iterrows():
        u, v = row['x_name'], row['y_name']
        if node_degrees[u] < MAX_PPI_DEGREE or node_degrees[v] < MAX_PPI_DEGREE:
            keep_indices.append(idx)
            node_degrees[u] += 1
            node_degrees[v] += 1
            
    ppi_pruned = ppi_full.loc[keep_indices]
    print(f"   ✂️  剪枝后 PPI: {len(ppi_pruned)} 条 (瘦身成功！)")

    # ================= 步骤 D: 合并保存 =================
    final_df = pd.concat([core_df, ppi_pruned]).drop_duplicates()
    
    print(f"\n🔹 5. 保存最终版图谱至: {OUTPUT_PATH}")
    final_df.to_csv(OUTPUT_PATH, index=False)
    
    print("-" * 50)
    print(f"🎉 最终条目数: {len(final_df)}")
    print(f"   构成: 核心知识 {len(core_df)} + 骨架PPI {len(ppi_pruned)}")
    print("-" * 50)

if __name__ == "__main__":
    main()

🔹 1. 加载 PrimeKG...
🔹 2. 执行清洗 (含新增的细胞组分黑名单)...
   🧹 基础清洗后: 3870350
🔹 3. 筛选 NACC 核心特征...
   ✅ 核心知识 (AD/代谢/精神): 37506 条

🔹 4. 补充基因互作 (限制每个基因最多 5 条边)...
   🕸  原始全量 PPI: 59888 条 (太多了，准备剪枝)
   ✂️  剪枝后 PPI: 11425 条 (瘦身成功！)

🔹 5. 保存最终版图谱至: primekg_nacc_base.csv
--------------------------------------------------
🎉 最终条目数: 48906
   构成: 核心知识 37506 + 骨架PPI 11425
--------------------------------------------------


In [5]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
from collections import Counter

# ================= 1. 配置区 =================
# NACC 本地数据文件
CSV_FILES = ['NACC_ad.csv', 'NACC_mci.csv', 'NACC_normal.csv']

# 指向 Step 2 生成的精简版底座
PRIMEKG_PATH = "primekg_nacc_base.csv"

# 输出文件
OUTPUT_TRIPLETS = "nacc_knowledge_triplets.csv"
OUTPUT_E2ID = "nacc_kg_entity2id.json"
OUTPUT_R2ID = "nacc_kg_relation2id.json"

# ================= 2. 映射逻辑定义 =================

# 基础人口学映射
base_col_map = {
    "age": "Concept:Age",
    "gender": "Concept:Sex",
    "education": "Concept:Education"
}

# [NACC特有] 病史字段映射 -> 对应 PrimeKG 里的关键词
# 逻辑：如果 his_HYPERTEN == 1，我们就去 PrimeKG 里找带 "Hypertension" 的节点连上
HISTORY_MAP = {
    "his_HYPERTEN": ["Hypertension"],
    "his_HYPERCHO": ["Hypercholesterolemia", "Hyperlipidemia", "Cholesterol"],
    "his_DIABETES": ["Diabetes", "Diabetes mellitus"],
    "his_DEP2YRS":  ["Depression", "Depressive"],
    "his_THYROID":  ["Hypothyroidism", "Thyroid"],
    "his_ANXIETY":  ["Anxiety"]
}

# 全局变量，用于缓存 PrimeKG 里的真实节点名，防止连到不存在的节点
VALID_KG_NODES = set()

def load_primekg():
    """读取筛选后的 PrimeKG 生物学子图"""
    print(f"🔹 1. 正在读取 PrimeKG 生物底座: {PRIMEKG_PATH} ...")
    if not os.path.exists(PRIMEKG_PATH):
        print(f"❌ 错误：找不到 {PRIMEKG_PATH}，请先运行 Step 2 代码")
        return []

    triplets = []
    global VALID_KG_NODES
    
    try:
        df = pd.read_csv(PRIMEKG_PATH, low_memory=False)
        
        # 自动识别列名
        cols = df.columns
        x_col = next((c for c in cols if 'x_name' in c or 'head' in c), 'x_name')
        y_col = next((c for c in cols if 'y_name' in c or 'tail' in c), 'y_name')
        r_col = next((c for c in cols if 'relation' in c), 'relation')

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing PrimeKG"):
            # 统一加上 PrimeKG 前缀
            h_raw = str(row[x_col])
            t_raw = str(row[y_col])
            
            h = "PrimeKG:" + h_raw
            t = "PrimeKG:" + t_raw
            r = str(row[r_col])
            
            triplets.append((h, r, t))
            
            # 存入缓存，供临床数据匹配使用
            VALID_KG_NODES.add(h_raw.lower())
            VALID_KG_NODES.add(t_raw.lower())

        print(f"   ✅ 已加载生物医学知识: {len(triplets)} 条")
        print(f"      (底座中包含 {len(VALID_KG_NODES)} 个唯一生物节点，准备用于临床连接)")
        
    except Exception as e:
        print(f"   ❌ 读取 PrimeKG 失败: {e}")
    return triplets

def find_best_kg_match(keywords):
    """在底座节点中寻找最匹配的节点名"""
    # 优先找完全匹配，其次找包含匹配
    # 这里我们简单一点，只要 VALID_KG_NODES 里有包含 keyword 的就行
    for kw in keywords:
        kw_lower = kw.lower()
        # 1. 精确匹配尝试
        if kw_lower in VALID_KG_NODES:
            return f"PrimeKG:{kw}" # 注意这里大小写可能不完美，但在 Embedding 里 id 对上就行
            
        # 2. 模糊搜索 (从缓存里找一个包含 kw 的)
        # 这是一个耗时操作，实际工程中可以优化，但 NACC 数据量不大，可以接受
        for node in VALID_KG_NODES:
            if kw_lower in node:
                # 找到第一个匹配的就返回 (比如找到 'Benign hypertension')
                # 恢复首字母大写的一般格式，虽然不严格
                return f"PrimeKG:{node.capitalize()}"
    return None

def process_nacc_files():
    """处理 NACC CSV 文件"""
    print(f"🔹 2. 正在处理 NACC 临床数据...")
    local_triplets = []
    patient_ids = set()
    
    # 统计计数器
    stats = {
        "patients": 0,
        "apoe_carriers": 0,
        "cog_scores": 0,
        "history_links": 0  # 记录有多少次成功把病史连到了 PrimeKG
    }
    
    # 具体的病史连接统计
    hist_breakdown = {k: 0 for k in HISTORY_MAP.keys()}

    for file_name in CSV_FILES:
        if not os.path.exists(file_name):
            print(f"   ⚠️ 跳过缺失文件: {file_name}")
            continue

        print(f"      -> 正在解析: {file_name} ...")
        df = pd.read_csv(file_name)

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"   Reading {file_name}"):
            # 1. 构建病人 ID (NACC 使用 'ID' 列)
            if 'ID' in row and pd.notna(row['ID']):
                p_id_raw = str(row['ID']).strip()
                patient_id = f"Patient:{p_id_raw}"
                patient_ids.add(patient_id)
                stats["patients"] += 1
            else:
                continue

            # 2. 基础属性 (Age, Sex, Education)
            for col, relation_name in base_col_map.items():
                if col in row and pd.notna(row[col]):
                    val = str(row[col])
                    target_node = f"{relation_name}:{val}"
                    local_triplets.append((patient_id, "has_attribute", target_node))

            # 3. 核心基因 (APOE)
            if 'apoe' in row and pd.notna(row['apoe']):
                try:
                    apoe_val = float(row['apoe'])
                    # NACC: 1=Yes (e4 carrier), 0=No
                    if apoe_val == 1.0: 
                        gene_node = "Gene:APOE_e4_Carrier"
                        local_triplets.append((patient_id, "has_gene_risk", gene_node))
                        # 桥接 PrimeKG
                        local_triplets.append((gene_node, "risk_factor_for", "PrimeKG:Alzheimer disease"))
                        local_triplets.append((gene_node, "is_variant_of", "PrimeKG:APOE")) 
                        stats["apoe_carriers"] += 1
                except: pass

            # 4. 认知评分 (MMSE, MoCA, CDR)
            # MMSE
            if 'mmse' in row and pd.notna(row['mmse']):
                local_triplets.append((patient_id, "has_mmse_score", f"Test:MMSE:{int(float(row['mmse']))}"))
                stats["cog_scores"] += 1
            
            # MoCA (NACC 特有)
            if 'moca' in row and pd.notna(row['moca']):
                local_triplets.append((patient_id, "has_moca_score", f"Test:MoCA:{int(float(row['moca']))}"))
                
            # CDR (Global)
            if 'cdr' in row and pd.notna(row['cdr']):
                local_triplets.append((patient_id, "has_cdr_score", f"Test:CDR:{float(row['cdr'])}"))

            # 5. [重头戏] 病史映射 -> 动态连接 PrimeKG
            for his_col, keywords in HISTORY_MAP.items():
                if his_col in row and pd.notna(row[his_col]):
                    try:
                        val = int(float(row[his_col]))
                        if val == 1: # 1 表示有该病史
                            # 尝试在底座里找对应的节点
                            target_kg_node = find_best_kg_match(keywords)
                            
                            if target_kg_node:
                                # 建立连接：病人 -> has_history -> PrimeKG:Hypertension
                                local_triplets.append((patient_id, "has_history", target_kg_node))
                                stats["history_links"] += 1
                                hist_breakdown[his_col] += 1
                            else:
                                # 如果底座里没找到，就创建一个占位符，保证信息不丢
                                fallback_node = f"Condition:{keywords[0]}"
                                local_triplets.append((patient_id, "has_history", fallback_node))
                    except: pass

    print(f"   ✅ NACC 处理完成: 生成 {len(local_triplets)} 条临床关系")
    print(f"      统计摘要:")
    print(f"      - 解析病人: {stats['patients']}")
    print(f"      - APOE携带者: {stats['apoe_carriers']}")
    print(f"      - 认知评分记录: {stats['cog_scores']}")
    print(f"      - 成功构建病史连接: {stats['history_links']} 条 (Bridge to PrimeKG)")
    print(f"      - 病史连接详情: {hist_breakdown}")
    
    return local_triplets

def analyze_graph_statistics(df):
    """打印详细的图谱体检报告 (增强版)"""
    print("\n" + "="*60)
    print("📊 [最终异构图谱深度体检报告]")
    print("="*60)
    
    # 1. 节点类型分析
    all_nodes = list(df['head']) + list(df['tail'])
    unique_nodes = set(all_nodes)
    
    node_types = []
    for n in unique_nodes:
        n_str = str(n)
        if n_str.startswith("Patient:"): node_types.append("Patient (病人)")
        elif n_str.startswith("PrimeKG:"): node_types.append("PrimeKG (生物知识)")
        elif n_str.startswith("Gene:"): node_types.append("Gene (风险基因)")
        elif n_str.startswith("Test:"): node_types.append("Test (认知量表)")
        elif n_str.startswith("Concept:"): node_types.append("Concept (人口学)")
        elif n_str.startswith("Condition:"): node_types.append("Condition (未匹配病史)")
        else: node_types.append("Other (其他)")
        
    type_counts = Counter(node_types)
    print(f"1. 节点构成 (总数: {len(unique_nodes)}):")
    for ntype, count in type_counts.most_common():
        print(f"   - {ntype:<20} : {count}")
        
    # 2. 关系分布
    print(f"\n2. 关系分布 (Top 15):")
    print(df['relation'].value_counts().head(15))
    
    # 3. 桥梁检查 (NACC 特别版)
    print(f"\n3. 关键连接检查:")
    
    # 检查 APOE 连接
    bridge_mask = (df['head'].str.contains("Patient")) & (df['tail'].str.contains("APOE"))
    bridge_count = len(df[bridge_mask])
    print(f"   - 病人 -> APOE 基因连接数: {bridge_count} 条")
    
    # 检查病史 -> PrimeKG 的连接
    # 比如 Patient -> has_history -> PrimeKG:Hypertension
    history_mask = (df['relation'] == 'has_history') & (df['tail'].str.startswith("PrimeKG:"))
    history_count = len(df[history_mask])
    print(f"   - 病人 -> PrimeKG病理节点 (Hypertension等) 连接数: {history_count} 条 (核心指标!)")

    # 检查是否有孤立病人
    # 这里不做复杂图计算，简单看是否有病人没有任何边 (一般不会，除非没数据)
    
    print("="*60 + "\n")

def main():
    # 1. 加载底座
    kg_triplets = load_primekg()
    
    # 2. 处理临床数据 (会利用底座信息进行匹配)
    nacc_triplets = process_nacc_files()

    if not nacc_triplets:
        print("❌ 严重警告：未生成任何 NACC 临床数据！")
        return

    # 3. 合并
    all_triplets = kg_triplets + nacc_triplets

    # 4. 保存与去重
    df_out = pd.DataFrame(all_triplets, columns=['head', 'relation', 'tail'])

    before_dedup = len(df_out)
    df_out.drop_duplicates(inplace=True)
    after_dedup = len(df_out)

    print(f"✂️  执行去重操作: {before_dedup} -> {after_dedup} (移除 {before_dedup - after_dedup})")

    # 5. 体检
    analyze_graph_statistics(df_out)

    # 6. 保存
    df_out.to_csv(OUTPUT_TRIPLETS, index=False)
    print(f"💾 最终 NACC 异构图谱已保存至: {OUTPUT_TRIPLETS}")

    # 7. 生成 ID 映射
    print("🔹 正在生成 Entity/Relation ID 映射表...")
    entities = sorted(list(set(df_out['head']) | set(df_out['tail'])))
    relations = sorted(list(set(df_out['relation'])))

    ent2id = {e: i for i, e in enumerate(entities)}
    rel2id = {r: i for i, r in enumerate(relations)}

    with open(OUTPUT_E2ID, 'w') as f: json.dump(ent2id, f)
    with open(OUTPUT_R2ID, 'w') as f: json.dump(rel2id, f)

    print(f"   映射表已保存: {OUTPUT_E2ID}, {OUTPUT_R2ID}")
    
    # 验证一个 ID
    sample_id = "NACC583163" # 来自你的 AD.csv 样例
    count = sum(1 for e in entities if sample_id in e)
    print(f"\n🔍 验证环节: 检查是否有示例病人 ({sample_id})")
    print(f"   包含该病人? {'✅ 是' if count > 0 else '❌ 否'}")

if __name__ == "__main__":
    main()

🔹 1. 正在读取 PrimeKG 生物底座: primekg_nacc_base.csv ...


Parsing PrimeKG: 100%|████████████████████████████████████████████████████████| 48906/48906 [00:04<00:00, 11578.32it/s]


   ✅ 已加载生物医学知识: 48906 条
      (底座中包含 9166 个唯一生物节点，准备用于临床连接)
🔹 2. 正在处理 NACC 临床数据...
      -> 正在解析: NACC_ad.csv ...


   Reading NACC_ad.csv: 100%|██████████████████████████████████████████████████████| 713/713 [00:00<00:00, 2737.18it/s]


      -> 正在解析: NACC_mci.csv ...


   Reading NACC_mci.csv: 100%|█████████████████████████████████████████████████████| 879/879 [00:00<00:00, 2980.84it/s]


      -> 正在解析: NACC_normal.csv ...


   Reading NACC_normal.csv: 100%|████████████████████████████████████████████████| 2116/2116 [00:00<00:00, 3352.61it/s]


   ✅ NACC 处理完成: 生成 28842 条临床关系
      统计摘要:
      - 解析病人: 3708
      - APOE携带者: 1122
      - 认知评分记录: 3645
      - 成功构建病史连接: 4808 条 (Bridge to PrimeKG)
      - 病史连接详情: {'his_HYPERTEN': 1388, 'his_HYPERCHO': 1553, 'his_DIABETES': 381, 'his_DEP2YRS': 753, 'his_THYROID': 503, 'his_ANXIETY': 230}
✂️  执行去重操作: 77748 -> 75505 (移除 2243)

📊 [最终异构图谱深度体检报告]
1. 节点构成 (总数: 13087):
   - PrimeKG (生物知识)       : 9211
   - Patient (病人)         : 3708
   - Concept (人口学)        : 102
   - Test (认知量表)          : 65
   - Gene (风险基因)          : 1

2. 关系分布 (Top 15):
relation
protein_protein               11698
disease_phenotype_positive    11527
has_attribute                 11106
bioprocess_protein             8670
disease_protein                7402
has_history                    4808
has_cdr_score                  3696
has_mmse_score                 3645
bioprocess_bioprocess          3628
has_moca_score                 2221
disease_disease                2182
pathway_protein                1718
has_gene_risk

In [6]:
# 知识图谱剪枝与优化 (NACC版)
import pandas as pd
import networkx as nx
import os
import shutil

# ================= ⚡ 配置区 =================
# 目标文件 (对应 Step 3 的输出)
TARGET_FILE = 'nacc_knowledge_triplets.csv'

# 保护名单：这些前缀的节点绝对不能删，哪怕它们只有一条连线
# 我们要把 NACC 特有的前缀加进去
PROTECTED_PREFIXES = [
    "Patient:",   # 病人 (核心)
    "Gene:",      # 风险基因 (桥梁)
    "Concept:",   # 年龄/性别/教育
    "Test:",      # MMSE/MoCA/CDR
    "Condition:"  # [新增] 那些没匹配上 PrimeKG 但依然保留的病史占位符
]

def analyze_graph(df, stage="清洗前"):
    """📊 打印图谱健康状况"""
    G = nx.Graph()
    G.add_edges_from(df[['head', 'tail']].values)
    
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    
    # 计算连通子图数量 (理想情况下应该很少，说明知识都连在一起)
    components = list(nx.connected_components(G))
    num_components = len(components)
    largest_comp = len(max(components, key=len)) if components else 0
    
    print(f"--- 🏥 [{stage}] 体检报告 ---")
    print(f"   节点: {num_nodes} | 边: {num_edges}")
    print(f"   连通孤岛数: {num_components} (越少越好)")
    print(f"   最大连通子图节点数: {largest_comp}")
    return G

def main():
    if not os.path.exists(TARGET_FILE):
        print(f"❌ 找不到文件: {TARGET_FILE}")
        return

    print(f"🚀 正在加载 {TARGET_FILE} ...")
    try:
        df = pd.read_csv(TARGET_FILE)
    except Exception as e:
        print(f"❌ 读取失败: {e}")
        return
    
    # 1. 术前体检
    analyze_graph(df, "清洗前")
    
    # 2. 创建备份 (安全第一)
    backup_file = TARGET_FILE + ".bak"
    shutil.copy(TARGET_FILE, backup_file)
    print(f"📦 已创建备份文件: {backup_file} (如果出问题可以用这个恢复)")

    # ================= 核心清洗逻辑 =================
    print("\n🧹 开始执行原地净化...")
    
    # 构建图结构
    G = nx.Graph()
    G.add_edges_from(df[['head', 'tail']].values)
    
    nodes_to_keep = set()
    
    # --- 策略 A: 移除无法到达病人的“死知识” ---
    # 逻辑：如果一个连通子图里连一个病人都没有，那这个子图就是垃圾数据
    print("   1. 正在切除与病人断连的孤立知识岛屿...")
    components = nx.connected_components(G)
    
    removed_islands = 0
    for comp in components:
        # 检查这个岛屿里有没有病人
        has_patient = any(str(n).startswith("Patient:") for n in comp)
        # 检查有没有关键基因，防止误删
        has_gene = any(str(n).startswith("Gene:") for n in comp)
        
        if has_patient or has_gene:
            nodes_to_keep.update(comp)
        else:
            removed_islands += 1
            
    print(f"      -> 移除了 {removed_islands} 个无效孤岛")
            
    # --- 策略 B: 修剪末梢悬挂节点 (Dead Ends) ---
    print("   2. 正在修剪低效的末梢节点...")
    # 只在保留下来的节点里修剪
    G_sub = G.subgraph(list(nodes_to_keep)).copy()
    
    nodes_to_remove = []
    for node in G_sub.nodes():
        # 如果是受保护节点，跳过
        if any(str(node).startswith(p) for p in PROTECTED_PREFIXES):
            continue
            
        # 如果度为1 (悬挂)，且不是受保护节点 -> 视为噪声
        # 这通常是 PrimeKG 里某个偏僻的蛋白，只连了一个上级，没连其他任何东西
        if G_sub.degree(node) == 1:
            nodes_to_remove.append(node)
            
    final_nodes = set(G_sub.nodes()) - set(nodes_to_remove)
    print(f"      -> 标记了 {len(nodes_to_remove)} 个悬挂节点准备删除")
    
    # ================= 保存结果 =================
    print("   3. 正在重组数据...")
    
    # 筛选只包含 final_nodes 的三元组
    mask = df['head'].isin(final_nodes) & df['tail'].isin(final_nodes)
    df_clean = df[mask].copy()
    
    # 3. 术后体检
    analyze_graph(df_clean, "清洗后")
    
    # 覆盖原文件
    df_clean.to_csv(TARGET_FILE, index=False)
    print(f"\n✅ 成功！原文件 {TARGET_FILE} 已更新。")
    print(f"   (原数据量 {len(df)} -> 现数据量 {len(df_clean)})")
    print("👉 你现在可以运行 DistMult 训练脚本了！")

if __name__ == "__main__":
    main()

🚀 正在加载 nacc_knowledge_triplets.csv ...
--- 🏥 [清洗前] 体检报告 ---
   节点: 13087 | 边: 54557
   连通孤岛数: 90 (越少越好)
   最大连通子图节点数: 12713
📦 已创建备份文件: nacc_knowledge_triplets.csv.bak (如果出问题可以用这个恢复)

🧹 开始执行原地净化...
   1. 正在切除与病人断连的孤立知识岛屿...
      -> 移除了 89 个无效孤岛
   2. 正在修剪低效的末梢节点...
      -> 标记了 2963 个悬挂节点准备删除
   3. 正在重组数据...
--- 🏥 [清洗后] 体检报告 ---
   节点: 9750 | 边: 51269
   连通孤岛数: 1 (越少越好)
   最大连通子图节点数: 9750

✅ 成功！原文件 nacc_knowledge_triplets.csv 已更新。
   (原数据量 75505 -> 现数据量 68927)
👉 你现在可以运行 DistMult 训练脚本了！


In [7]:
# train_nacc_distmult.py
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
import json
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

# ================= ⚡ 配置区 (NACC 版) =================
# 1. 输入文件 (对应 Step 4 剪枝后的输出)
TRIPLETS_FILE = 'nacc_knowledge_triplets.csv'

# 2. 映射表输出 (我们将重新生成这些文件，确保与剪枝后的数据完全对应)
ENTITY2ID_FILE = 'nacc_kg_entity2id.json'
RELATION2ID_FILE = 'nacc_kg_relation2id.json'

# 3. 输出文件 (训练好的 Embeddings)
OUTPUT_EMBED = 'nacc_kg_embeddings.npy'

# 4. 训练超参数 (保持标准设置)
EMBED_DIM = 128
NUM_EPOCHS = 100
BATCH_SIZE = 512
LR = 0.005
TRAIN_RATIO = 0.95  # 留 5% 做验证即可
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 [NACC] 训练设备: {DEVICE}")

# ================= 🛠️ 辅助函数: 重建 ID 映射 =================
def rebuild_mappings(df):
    """
    因为剪枝步骤删除了很多节点，旧的 json 映射表已经不准了。
    为了保证 Embedding 矩阵紧凑，我们需要根据最终的 CSV 重新生成映射。
    """
    print("🔄 正在基于最终数据重建 ID 映射表...")
    
    # 获取所有唯一的头实体和尾实体
    entities = sorted(list(set(df['head']) | set(df['tail'])))
    relations = sorted(list(set(df['relation'])))
    
    ent2id = {e: i for i, e in enumerate(entities)}
    rel2id = {r: i for i, r in enumerate(relations)}
    
    print(f"   - 实体总数: {len(entities)} (原计划 13087 -> 现 {len(entities)})")
    print(f"   - 关系总数: {len(relations)}")
    
    # 保存新的映射表 (覆盖旧的)
    with open(ENTITY2ID_FILE, 'w') as f: json.dump(ent2id, f)
    with open(RELATION2ID_FILE, 'w') as f: json.dump(rel2id, f)
    
    return ent2id, rel2id

# ================= 🛠️ 数据加载类 =================
class KGDataset(Dataset):
    def __init__(self, df, entity2id, relation2id):
        self.triplets = []
        skipped = 0

        # 将字符串转换为 ID
        # 使用 numpy 向量化操作加速 (如果数据量巨大)，这里用循环足够
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Indexing Data"):
            try:
                h = entity2id.get(str(row['head']).strip())
                r = relation2id.get(str(row['relation']).strip())
                t = entity2id.get(str(row['tail']).strip())

                if h is not None and r is not None and t is not None:
                    self.triplets.append((h, r, t))
                else:
                    skipped += 1
            except Exception:
                skipped += 1
                continue

        self.triplets = torch.LongTensor(self.triplets)
        print(f"    📊 最终有效三元组: {len(self.triplets)}")

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        return self.triplets[idx]

# ================= 🧠 DistMult 模型 =================
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, embed_dim):
        super(DistMult, self).__init__()
        self.num_entities = num_entities
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        self.rel_emb = nn.Embedding(num_relations, embed_dim)

        # Xavier 初始化有助于收敛
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

        self.criterion = nn.MarginRankingLoss(margin=1.0)

    def forward(self, h, r, t):
        h_e = self.ent_emb(h)
        r_e = self.rel_emb(r)
        t_e = self.ent_emb(t)
        # DistMult 核心评分函数: <h, r, t> = sum(h * r * t)
        score = torch.sum(h_e * r_e * t_e, dim=1)
        return score

    def calculate_loss(self, h, r, t):
        batch_size = h.size(0)
        # 负采样: 随机替换尾实体
        neg_t = torch.randint(0, self.num_entities, (batch_size,), device=h.device)
        
        pos_score = self.forward(h, r, t)
        neg_score = self.forward(h, r, neg_t)
        
        # 让正样本得分比负样本高至少 1.0
        target = torch.ones(batch_size, device=h.device)
        loss = self.criterion(pos_score, neg_score, target)
        return loss

# ================= 🏃 主训练循环 =================
def train():
    if not os.path.exists(TRIPLETS_FILE):
        print(f"❌ 找不到 {TRIPLETS_FILE}，请检查路径")
        return

    # 1. 读取数据
    print(f"📥 正在读取: {TRIPLETS_FILE} ...")
    df = pd.read_csv(TRIPLETS_FILE)
    
    # 2. 重建 ID 映射 (确保 Embedding 矩阵紧凑)
    entity2id, relation2id = rebuild_mappings(df)
    
    num_ents = len(entity2id)
    num_rels = len(relation2id)

    # 3. 准备 Dataset
    dataset = KGDataset(df, entity2id, relation2id)

    train_size = int(TRAIN_RATIO * len(dataset))
    test_size = len(dataset) - train_size
    train_data, test_data = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

    # 4. 初始化模型
    model = DistMult(num_ents, num_rels, EMBED_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    print(f"\n🔥 [NACC] 开始训练 (共 {NUM_EPOCHS} 轮)...")

    # 5. 训练循环
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        
        # 进度条
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}", leave=False)

        for batch in progress:
            batch = batch.to(DEVICE)
            h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

            optimizer.zero_grad()
            loss = model.calculate_loss(h, r, t)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)

        # 每轮简单验证一下
        if (epoch + 1) % 1 == 0:
            model.eval()
            test_loss = 0
            with torch.no_grad():
                for batch in test_loader:
                    batch = batch.to(DEVICE)
                    h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
                    loss = model.calculate_loss(h, r, t)
                    test_loss += loss.item()
            avg_test = test_loss / len(test_loader) if len(test_loader) > 0 else 0
            print(f"Epoch {epoch + 1:03d} | 📉 Train Loss: {avg_loss:.4f} | 🔍 Test Loss: {avg_test:.4f}")

    # 6. 保存结果
    print("\n" + "=" * 40)
    print(f"💾 训练完成！正在保存 NACC Embeddings 至: {OUTPUT_EMBED}")

    embeddings = model.ent_emb.weight.detach().cpu().numpy()
    np.save(OUTPUT_EMBED, embeddings)

    print(f"✅ 保存成功！矩阵形状: {embeddings.shape}")
    print(f"   (行数应为 {num_ents}，列数应为 {EMBED_DIM})")
    print("=" * 40)
    print("👉 下一步：你可以使用这些 embeddings 训练你的分类器了！")
    print("   记得加载 nacc_kg_entity2id.json 来查找病人对应的行号。")

if __name__ == "__main__":
    train()

🚀 [NACC] 训练设备: cpu
📥 正在读取: nacc_knowledge_triplets.csv ...
🔄 正在基于最终数据重建 ID 映射表...
   - 实体总数: 9750 (原计划 13087 -> 现 9750)
   - 关系总数: 20


Indexing Data: 100%|██████████████████████████████████████████████████████████| 68927/68927 [00:06<00:00, 11481.64it/s]


    📊 最终有效三元组: 68915

🔥 [NACC] 开始训练 (共 100 轮)...


Epoch 001 | 📉 Train Loss: 0.8080 | 🔍 Test Loss: 0.3120


Epoch 002 | 📉 Train Loss: 0.0935 | 🔍 Test Loss: 0.0693


...

Epoch 096 | 📉 Train Loss: 0.0063 | 🔍 Test Loss: 0.1048


Epoch 097 | 📉 Train Loss: 0.0077 | 🔍 Test Loss: 0.0990


Epoch 098 | 📉 Train Loss: 0.0064 | 🔍 Test Loss: 0.1176


Epoch 099 | 📉 Train Loss: 0.0050 | 🔍 Test Loss: 0.0996


Epoch 100 | 📉 Train Loss: 0.0057 | 🔍 Test Loss: 0.1036

💾 训练完成！正在保存 NACC Embeddings 至: nacc_kg_embeddings.npy
✅ 保存成功！矩阵形状: (9750, 128)
   (行数应为 9750，列数应为 128)
👉 下一步：你可以使用这些 embeddings 训练你的分类器了！
   记得加载 nacc_kg_entity2id.json 来查找病人对应的行号。
